# Install Requirements

In [1]:
!pip install dateparser selenium tqdm

  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 1.4 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/9.6 MB 1.4 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.6 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/9.6 MB 2.6 MB/s eta 0:00:03
   ------------- -------------------------- 3.1/9.6 MB 2.8 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.6 MB 3.1 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.6 MB 3.4 MB/s eta 0:00:02
   ------------------------- -------------- 6.0/9.6 MB 3.5 MB/s eta 0:00:02
   ----------------------------- ---------- 7.1/9.6 MB 3.6 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.6 MB 


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement python-dateparser==1.2.0 (from versions: none)

[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for python-dateparser==1.2.0


# Scraping (Gemini, kode belum lengkap)

In [ ]:
import csv
import logging
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dateparser import parse
from selenium import webdriver
from selenium.common.exceptions import (StaleElementReferenceException,
                                        TimeoutException)
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from tqdm import tqdm

# Configuration
CONFIG = {
    'input_csv_path': 'batch4.csv',
    'output_csv_path': 'olx_housing_dataset_final.csv',
    'log_file_path': 'scraper.log',
    'max_threads': 10,  # Reduced for stability, can be adjusted
    'url_column_index': 4,
    'timeout': 20 # Increased timeout for page loads
}

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - [%(threadName)s] - %(message)s',
    handlers=[
        logging.FileHandler(CONFIG['log_file_path']),
        logging.StreamHandler()
    ]
)

def extract_id_from_url(url):
    pattern = r'-iid-(\d+)$'
    match = re.search(pattern, url)
    return match.group(1) if match else None

def get_nested_value(data_dict, keys, default=None):
    for key in keys:
        if isinstance(data_dict, list):
            try:
                data_dict = data_dict[key]
            except (IndexError, TypeError):
                return default
        else:
            data_dict = data_dict.get(key, default)
            if data_dict is default:
                return default
    return data_dict or default

def get_data(app_data, ads_ID, data_type, key_name=None, location_type=None):
    if data_type == "location" and location_type:
        address_components = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "metadata", "locations", 0, "tree", 0, "addressComponents"], [])
        for component in address_components:
            if component.get("type") == location_type:
                return component.get("name", None)
        return None
    elif data_type == "images":
        images = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "images"], [])
        return [image.get("url") for image in images if 'url' in image]
    elif data_type == "parameters" and key_name:
        parameters = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "parameters"], [])
        for param in parameters:
            if param.get("key_name").lower() == key_name.lower():
                if param.get("type") == "single":
                    value = param.get("value_name", None)
                    if key_name.lower() == "sertifikasi":
                        if value and "shm" in value.lower():
                            return "SHM"
                        return None
                    return value
                elif param.get("type") == "multi":
                    values = [value.get("value_name") for value in param.get("values", [])]
                    return values[0] if len(values) == 1 else values
        return None
    return None

def get_seller_name(app_data, ads_ID):
    seller_id = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "user_id"])
    return get_nested_value(app_data, ["states", "users", "elements", seller_id, "name"])

def extract_app_data(driver):
    retries = 3
    for _ in range(retries):
        try:
            script_tags = driver.find_elements(By.TAG_NAME, 'script')
            for script in script_tags:
                script_content = script.get_attribute('innerHTML')
                if 'window.__APP' in script_content:
                    driver.execute_script(script_content)
                    return driver.execute_script("return window.__APP;")
        except StaleElementReferenceException:
            logging.warning("Stale element reference, retrying...")
            time.sleep(1)
            continue
    logging.error("Stale element, retries exhausted")
    return None

def split_sentences(description):
    sentences = description.splitlines()
    return [line.strip() for line in sentences if line.strip()]
    
def scrape_description(ads_description, kata_kunci_list, entity_type=None):
    headings = ["Timur", "Tenggara", "Selatan", "Barat Daya", "Barat", "Barat Laut", "Utara", "Timur Laut"]
    if not isinstance(ads_description, list): return None
    for kata_kunci in kata_kunci_list:
        if entity_type == 'electricity': pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*([\d.,]+)\s*(watt|va|kva|token|w|wt|kwh)?'
        elif entity_type in ['garage', 'carport']: pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*(\d+)?\s*(mobil|mbl|cars?)?'
        elif entity_type == 'heading': pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*(.*?)(Timur|Tenggara|Selatan|Barat Daya|Barat Laut|Barat|Utara|Timur Laut|Kiblat|Khiblat)'
        else: pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*([\w\s\.,]+)'
        for sentence in ads_description:
            if isinstance(sentence, str) and kata_kunci.lower() in sentence.lower():
                match = re.search(pattern, sentence, re.IGNORECASE)
                if match:
                    if entity_type == 'electricity':
                        value_str = match.group(1).replace('.', '').replace(',', '')
                        return int(value_str)
                    elif entity_type in ['garage', 'carport']:
                        value_str = match.group(1)
                        return int(value_str) if value_str else 1
                    elif entity_type == 'heading':
                        heading = match.group(2).capitalize()
                        return 'Barat' if heading.lower() in ['kiblat', 'khiblat'] else heading
                    else:
                        return match.group(1).strip().split(" ")[0]
    return None

def count_rooms(ads_description, keyword_list):
    for keyword in keyword_list:
        pattern = rf'(\d+)?\s*{re.escape(keyword)}'
        for sentence in ads_description:
            match = re.search(pattern, sentence, re.IGNORECASE)
            if match:
                return int(match.group(1)) if match.group(1) else 1
    return 0

def extract_floors(ads_description, keyword_list):
    for keyword in keyword_list:
        pattern = rf'(\d+)\s*{re.escape(keyword)}|{re.escape(keyword)}\s*[\:\-\s]*\d+'
        for sentence in ads_description:
            match = re.search(pattern, sentence, re.IGNORECASE)
            if match:
                if match.group(1): return int(match.group(1))
                alt_match = re.search(rf'{re.escape(keyword)}\s*[\:\-\s]*(\d+)', sentence, re.IGNORECASE)
                if alt_match: return int(alt_match.group(1))
    return 1

def parse_date(ads_postingdate):
    if not ads_postingdate: return None, None, None
    date_obj = parse(ads_postingdate, languages=['id', 'en'])
    if date_obj:
        return f"{date_obj.day} {date_obj.strftime('%B')} {date_obj.year}", date_obj.month, date_obj.year
    return None, None, None
    
def extract_all_ads_info(ads_description):
    if not isinstance(ads_description, list) or not all(isinstance(s, str) for s in ads_description): ads_description = []
    keywords = {'garage': ['Garasi', 'Garage'],'carport': ['Carport', 'Carpot'],'electricity': ['Listrik', 'electricity', 'pln', 'listtrik'],'heading': ['Hadap', 'Orientasi'],'ruang_tamu': ['Ruang Tamu'],'ruang_makan': ['Ruang Makan'],'maid_bedroom': ['Kamar Pembantu'],'maid_bathroom': ['Kamar Mandi Pembantu'],'floors': ['Lantai']}
    info = {
        'garage': scrape_description(ads_description, keywords['garage'], 'garage') or 0,
        'carport': scrape_description(ads_description, keywords['carport'], 'carport') or 0,
        'electricity': scrape_description(ads_description, keywords['electricity'], 'electricity'),
        'heading': scrape_description(ads_description, keywords['heading'], 'heading'),
        'ruang_tamu': count_rooms(ads_description, keywords['ruang_tamu']),
        'ruang_makan': count_rooms(ads_description, keywords['ruang_makan']),
        'maid_bedroom': count_rooms(ads_description, keywords['maid_bedroom']),
        'maid_bathroom': count_rooms(ads_description, keywords['maid_bathroom']),
        'floors': extract_floors(ads_description, keywords['floors'])
    }
    other_rooms_count = sum(1 for s in ads_description if "ruang" in s.lower() and "ruang tamu" not in s.lower() and "ruang makan" not in s.lower())
    info['additional_rooms'] = 1 if other_rooms_count > 0 else 0
    return info

def create_driver():
    """Creates and returns a new Selenium WebDriver instance."""
    options = Options()
    options.add_argument("--inprivate")
    options.add_argument("--headless")  # Headless is better for automated scripts
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-logging")
    options.add_argument("log-level=3") # Suppress console logs
    return webdriver.Edge(options=options)

# The main scraping function accepts a driver instance
def scrape_data_from_url(driver, url):
    """Scrapes data from a single URL using a provided driver instance."""
    try:
        driver.get(url)
        # Exception handling
        WebDriverWait(driver, CONFIG['timeout']).until(
            EC.presence_of_element_located((By.TAG_NAME, "script"))
        )

        ads_ID = extract_id_from_url(url)
        if not ads_ID:
            logging.error(f"Failed to extract Ad ID from URL: {url}")
            return None

        app_data = extract_app_data(driver)
        if not app_data:
            logging.error(f"Failed to extract __APP data from URL: {url}")
            return None

        # Data Extraction Logic
        ads_title = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "title"])
        ads_price = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "price", "value", "raw"])
        ads_description_raw = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "description"])
        ads_description = split_sentences(ads_description_raw) if ads_description_raw else []
        ads_info = extract_all_ads_info(ads_description)
        ads_postingdate_raw = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "created_at"])
        ads_postingdate, ads_month, ads_year = parse_date(ads_postingdate_raw)
        ads_floors = get_data(app_data, ads_ID, "parameters", key_name="Lantai") or ads_info.get('floors')

        # Collect data into dictionary
        scraped_data = {
            'url': url, 'ads_id': ads_ID, 'title': ads_title, 'price': ads_price,
            'type': get_data(app_data, ads_ID, "parameters", key_name="Tipe"),
            'land_area': get_data(app_data, ads_ID, "parameters", key_name="Luas Tanah"),
            'building_area': get_data(app_data, ads_ID, "parameters", key_name="Luas Bangunan"),
            'bedrooms': get_data(app_data, ads_ID, "parameters", key_name="Kamar Tidur"),
            'bathrooms': get_data(app_data, ads_ID, "parameters", key_name="Kamar Mandi"),
            'maid_bedrooms': ads_info.get('maid_bedroom'), 'maid_bathrooms': ads_info.get('maid_bathroom'),
            'ruang_tamu': ads_info.get('ruang_tamu'), 'ruang_makan': ads_info.get('ruang_makan'),
            'additional_rooms': ads_info.get('additional_rooms'), 'floors': ads_floors,
            'certificate': get_data(app_data, ads_ID, "parameters", key_name="Sertifikasi"),
            'address': get_data(app_data, ads_ID, "parameters", key_name="Alamat Lokasi"),
            'address_city': get_data(app_data, ads_ID, "location", location_type="CITY"),
            'garage_capacity': ads_info.get('garage'), 'carport_capacity': ads_info.get('carport'),
            'facilities': get_data(app_data, ads_ID, "parameters", key_name="Fasilitas"),
            'description': ads_description, 'posting_date': ads_postingdate,
            'posting_date_month': ads_month, 'posting_date_year': ads_year,
            'poster': get_seller_name(app_data, ads_ID),
            'electricity_capacity': ads_info.get('electricity'),
            'house_orientation': ads_info.get('heading'),
            'image_url': get_data(app_data, ads_ID, "images")
        }
        logging.info(f"Successfully scraped: {url}")
        return scraped_data

    except TimeoutException:
        logging.error(f"Timeout while loading URL: {url}")
        return None
    except Exception as e:
        logging.error(f"An unexpected error occurred while scraping {url}: {e}", exc_info=True)
        return None

def scrape_worker(urls_chunk):
    """Worker function for a thread. Initializes one driver and scrapes a chunk of URLs."""
    driver = create_driver()
    results = []
    try:
        for url in urls_chunk:
            data = scrape_data_from_url(driver, url)
            if data:
                results.append(data)
    finally:
        driver.quit()
    return results

def main():
    logging.info("Scraper script started.")
    try:
        with open(CONFIG['input_csv_path'], mode='r', encoding='utf-8') as f:
            reader = csv.reader(f)
            # Skip header if exists
            # next(reader, None) 
            listing_urls = [row[CONFIG['url_column_index']] for row in reader if row]
    except FileNotFoundError:
        logging.error(f"Input file not found: {CONFIG['input_csv_path']}")
        return
    
    if not listing_urls:
        logging.warning("No URLs found in the input file.")
        return

    logging.info(f"Found {len(listing_urls)} URLs to scrape.")
    
    all_scraped_data = []
    with ThreadPoolExecutor(max_workers=CONFIG['max_threads']) as executor:
        # Submit all URLs to the executor
        future_to_url = {executor.submit(scrape_data_from_url, create_driver(), url): url for url in listing_urls}
        
        # Process results as they complete
        for future in tqdm(as_completed(future_to_url), total=len(listing_urls), desc="Scraping Progress"):
            try:
                data = future.result()
                if data:
                    all_scraped_data.append(data)
            except Exception as e:
                url = future_to_url[future]
                logging.error(f"Error processing result for {url}: {e}", exc_info=True)


    if not all_scraped_data:
        logging.warning("No data was scraped successfully.")
        return

    # Write to CSV
    try:
        with open(CONFIG['output_csv_path'], 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = all_scraped_data[0].keys()
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_scraped_data)
        logging.info(f"Successfully created CSV file: {CONFIG['output_csv_path']}")
    except IOError:
        logging.error(f"Could not write to CSV file: {CONFIG['output_csv_path']}")

if __name__ == "__main__":
    main()